### Use to search for samples taken on different PMT settings

In [4]:
from pathlib import Path
import re
import pandas as pd

In [5]:
SEARCH_DIR = Path("/Users/michaelstaiger/Desktop/gitRepos/IFCBParticleSize/EmpyricalAnalysis/IFCBData/nauset/nauset")   # change this
OUT_CSV = "hdr_master_table.csv"

In [7]:
def parse_datetime_from_text(text):
    """
    Try to find a timestamp in a string.
    Supports patterns like:
      D20200710T123456
      20200710T123456
      2020-07-10 12:34:56
    """
    patterns = [
        r'(D\d{8}T\d{6})',
        r'(\d{8}T\d{6})',
        r'(\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2})',
    ]
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            ts = pd.to_datetime(m.group(1), errors="coerce")
            if pd.notna(ts):
                return ts
    return pd.NaT


def extract_hdr_value(hdr_path):
    """
    Read a .hdr file and return:
      - filename
      - datetime
      - PMTtriggerSelection_DAQ_MCConly value
    """
    hdr_path = Path(hdr_path)

    # Try datetime from filename or path first
    dt = parse_datetime_from_text(hdr_path.name)
    if pd.isna(dt):
        dt = parse_datetime_from_text(str(hdr_path))

    value = None

    with open(hdr_path, "r", errors="ignore") as f:
        for line in f:
            if "PMTtriggerSelection_DAQ_MCConly" in line:
                # Expect something like:
                # PMTtriggerSelection_DAQ_MCConly: 2
                parts = line.split(":", 1)
                if len(parts) == 2:
                    raw = parts[1].strip()
                    try:
                        value = int(raw)
                    except ValueError:
                        try:
                            value = float(raw)
                        except ValueError:
                            value = raw
                break

            # Optional fallback if the datetime appears in the file contents
            if pd.isna(dt) and ("Date" in line or "Time" in line or "Datetime" in line):
                maybe_dt = parse_datetime_from_text(line)
                if pd.notna(maybe_dt):
                    dt = maybe_dt

    return {
        "FileName": hdr_path.name,
        "FilePath": str(hdr_path),
        "Datetime": dt,
        "PMTtriggerSelection_DAQ_MCConly": value,
    }

In [8]:
hdr_files = sorted(SEARCH_DIR.rglob("*.hdr"))

rows = []
for i, hdr_file in enumerate(hdr_files, start=1):
    try:
        rows.append(extract_hdr_value(hdr_file))
    except Exception as e:
        rows.append({
            "FileName": hdr_file.name,
            "FilePath": str(hdr_file),
            "Datetime": pd.NaT,
            "PMTtriggerSelection_DAQ_MCConly": None,
        })
        print(f"Failed on {hdr_file}: {e}")

    if i % 500 == 0:
        print(f"Processed {i}/{len(hdr_files)}")

hdr_df = pd.DataFrame(rows)
hdr_df

,FileName,FilePath,Datetime,PMTtriggerSelection_DAQ_MCConly
0,D20250501T170646_IFCB145.hdr,/Users/michaelstaiger/Desktop/gitRepos/IFCBPar...,2025-05-01 17:06:46,2
1,D20250502T000355_IFCB145.hdr,/Users/michaelstaiger/Desktop/gitRepos/IFCBPar...,2025-05-02 00:03:55,2
2,D20250502T135508_IFCB145.hdr,/Users/michaelstaiger/Desktop/gitRepos/IFCBPar...,2025-05-02 13:55:08,2
3,D20250503T000141_IFCB145.hdr,/Users/michaelstaiger/Desktop/gitRepos/IFCBPar...,2025-05-03 00:01:41,2
4,D20250503T160523_IFCB145.hdr,/Users/michaelstaiger/Desktop/gitRepos/IFCBPar...,2025-05-03 16:05:23,2
...,...,...,...,...
87,D20250630T135351_IFCB144.hdr,/Users/michaelstaiger/Desktop/gitRepos/IFCBPar...,2025-06-30 13:53:51,2
88,D20250702T004811_IFCB145.hdr,/Users/michaelstaiger/Desktop/gitRepos/IFCBPar...,2025-07-02 00:48:11,2
89,D20250702T121028_IFCB145.hdr,/Users/michaelstaiger/Desktop/gitRepos/IFCBPar...,2025-07-02 12:10:28,2
90,D20250703T001941_IFCB145.hdr,/Users/michaelstaiger/Desktop/gitRepos/IFCBPar...,2025-07-03 00:19:41,2


In [ ]:
## Cleaing and saving if needed
hdr_df = hdr_df.sort_values("Datetime").reset_index(drop=True)
#hdr_df.to_csv(OUT_CSV, index=False)

OUT_CSV, hdr_df.head()

In [9]:
hdr_df["PrevValue"] = hdr_df["PMTtriggerSelection_DAQ_MCConly"].shift(1)
change_df = hdr_df.loc[
    hdr_df["PMTtriggerSelection_DAQ_MCConly"] != hdr_df["PrevValue"]
].copy()

change_df

,FileName,FilePath,Datetime,PMTtriggerSelection_DAQ_MCConly,PrevValue
0,D20250501T170646_IFCB145.hdr,/Users/michaelstaiger/Desktop/gitRepos/IFCBPar...,2025-05-01 17:06:46,2,NaN
